## TRIGRAM Language Model

In [9]:
import torch
import requests
from pathlib import Path

In [1]:
## Download Dataset
with open("/Users/mac/Desktop/Machine Learning/DL/models/data/names.txt", "r") as f:
    words=f.read().splitlines()
    
print(f"Number of names:{len(words)}")
print(words[:10])

Number of names:32033
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [3]:
# Build Vocabulary
chars=sorted(list(set("".join(words))))
print(chars)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
## Create Character to index 
stoi={ch:i+1 for i, ch in enumerate(chars)}
stoi["."]=0

print(stoi)

# Create reverse mapping
itos={i:ch for ch, i in stoi.items()}

print(itos)

vocab_size=len(stoi)
print("Vocabulary size:", vocab_size)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
Vocabulary size: 27


In [10]:
# Create tensor
N3=torch.zeros(
    (vocab_size, vocab_size, vocab_size),
    dtype=torch.int32
)

for word in words:
    chars=[".", "."]+list(word)+["."]
    
    for ch1, ch2, ch3 in zip(chars, chars[1:], chars[2:]):
        i=stoi[ch1]
        j=stoi[ch2]
        k=stoi[ch3]
        
        N3[i, j, k]+=1
        

In [11]:
# Convert Counts to Probabilities
P3=(N3+1).float()

# normalize probabilities
P3/=P3.sum(dim=2, keepdim=True)

print(P3[0, 0].sum())

tensor(1.)


## Greedy Decoding

In [12]:
def generate_greedy(P3):
    i=0
    j=0
    
    name=[]
    
    while True:
        probs=P3[i, j]
        k=torch.argmax(probs).item()
        
        if k==0:
            break
        
        name.append(itos[k])
        
        i, j=j, k
    
    return "".join(name)

In [13]:
# Test
for _ in range(5):
    print(generate_greedy(P3))

alee
alee
alee
alee
alee


## Random Sampling


In [14]:
def generate_random(P3):
    i=0
    j=0
    
    name=[]
    while True:
        probs=P3[i, j]
        
        k=torch.multinomial(
            probs, num_samples=1
        ).item()
        
        if k==0:
            break
        
        name.append(itos[k])
        
        i, j=j, k
        
    return "".join(name)

In [15]:
for _ in range(10):
    print(generate_random(P3))

sophaddia
my
elyn
naily
tree
bre
senn
romadhyab
milunata
avonnatrindeenel


## Top-K Sampling

In [16]:
def generate_top_k(P3, k_top=5):
    i=0
    j=0
    
    name=[]
    
    while True:
        probs=P3[i, j]
        
        top_probs, top_indices=torch.topk(
            probs, k_top
        )
        top_probs=(
            top_probs/top_probs.sum()
        )
        choice=torch.multinomial(
            top_probs, 1
        ).item()
        
        k=top_indices[choice].item()
        
        if k==0:
            break
        
        name.append(itos[k])
        
        i, j=j, k
        
    return "".join(name)
            

In [17]:
# Test
for _ in range(10):
    print(generate_top_k(P3, k_top=5))

arishona
mila
mirosen
keirosin
alee
anne
sarley
jaya
sielly
kha
